<a href="https://colab.research.google.com/github/huimouqianxiao123/Peft-Qwen2.5-/blob/main/%E5%8A%A0%E8%BD%BD%E8%AE%AD%E7%BB%83%E6%95%B0%E6%8D%AE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 34.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.18.0
    Uninstalling huggingface_hub-1.18.0:
      Successfully uninstalled huggingface_hub-1.18.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.10.2
    Uninstalling transformers-5.10.2:
      Successfully uninstalled transformers-5.10.2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
model_path = "/content/drive/MyDrive/models/qwen2.5-7b-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
file_path = '/content/drive/MyDrive/data/intent_train.jsonl'
all_data = []

print("正在读取原始数据集...")
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            all_data.append(json.loads(line))

sample_size = min(500, len(all_data))
random.seed(42)  # 固定随机种子
sampled_dataset = random.sample(all_data, sample_size)
print(f"成功随机抽取 {sample_size} 条测试样本！")

正在读取原始数据集...
成功随机抽取 500 条测试样本！


In [ ]:
from tqdm import tqdm
# 提取你数据中的 17 个标准标签，用于后续对齐模型输出
intents_list = [
    "查询物流", "催促配送", "修改订单", "取消订单", "申请退款", "退换货",
    "发票咨询", "支付咨询", "价格价保", "优惠活动", "商品咨询", "催促处理",
    "投诉反馈", "转人工", "问候确认", "业务咨询", "其他"
]

# ==========================================
# 3. 执行批量推理测试
# ==========================================
y_true = []
y_pred = []

print("\n开始执行批量意图识别测试...")
for item in tqdm(sampled_dataset):
    # 直接读取你的标准字段
    instruction = item.get('instruction', '')
    # 清理掉输入文本末尾的换行符和多余逗号 \r,
    user_input = item.get('input', '').strip(' \r\n,')
    true_label = item.get('output', '')

    if not instruction or not true_label:
        continue

    # 构建 Qwen Chat 格式：把你的 instruction 当作系统提示词，input 当作用户输入
    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": f"需要判断的文本：【{user_input}】\n请输出最贴切的标签名："}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=15,
            temperature=0.1,  # 低温度，保持分类稳定性
            do_sample=False
        )

    # 提取生成的文本
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    # 结果后处理：在模型的回复中寻找命中的标准标签
    pred_label = "其他" # 默认 fallback
    for intent in intents_list:
        if intent in response:
            pred_label = intent
            break

    y_true.append(true_label)
    y_pred.append(pred_label)


开始执行批量意图识别测试...


  0%|          | 0/500 [00:00<?, ?it/s][transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 500/500 [05:41<00:00,  1.47it/s]


In [ ]:
from sklearn.metrics import classification_report, accuracy_score
print("\n================== 测试评估报告 ==================")
accuracy = accuracy_score(y_true, y_pred)
print(f"🎯 总体准确率 (Overall Accuracy): {accuracy:.2%}\n")

print("📊 各意图类别的详细指标 (Precision / Recall / F1-Score):")
print(classification_report(y_true, y_pred, labels=intents_list, zero_division=0))
print("==================================================")


================== 测试评估报告 ==================
🎯 总体准确率 (Overall Accuracy): 45.40%

📊 各意图类别的详细指标 (Precision / Recall / F1-Score):
              precision    recall  f1-score   support

        查询物流       0.21      0.11      0.14        75
        催促配送       0.05      1.00      0.10         3
        修改订单       0.83      0.71      0.77        14
        取消订单       0.50      0.93      0.65        15
        申请退款       0.55      0.78      0.65        27
         退换货       0.83      0.46      0.59        52
        发票咨询       0.55      0.75      0.63         8
        支付咨询       0.50      0.43      0.46         7
        价格价保       0.77      0.71      0.74        14
        优惠活动       0.17      0.20      0.18         5
        商品咨询       0.03      0.50      0.06         2
        催促处理       0.14      0.21      0.17        33
        投诉反馈       0.00      0.00      0.00         0
         转人工       0.00      0.00      0.00         0
        问候确认       0.78      0.83      0.80       105
       